# Campo Real Energia — executar o pipeline no Google Colab

Este notebook roda o projeto inteiro sem instalar nada no computador. Ele precisa de:

1. O **projeto** (o `.zip` do repositório ou o link do GitHub).
2. Um banco **PostgreSQL** acessível pela internet (ex.: Neon) com a **string de conexão**.
3. Opcional: o script da professora `campo_real_energia_postgres.sql`, caso a origem ainda não tenha sido criada no banco.

Execute as células em ordem (**Ambiente de execução → Executar tudo**).

## 1. Trazer o projeto
Se o repositório já estiver no GitHub, cole o link em `REPO_GITHUB`. Se deixar vazio, a célula pede o upload do `.zip`.

In [ ]:
import os, sys, zipfile
from google.colab import files

REPO_GITHUB = ""

os.chdir("/content")
if REPO_GITHUB:
    !rm -rf campo-real-dw
    !git clone -q {REPO_GITHUB} campo-real-dw
else:
    enviado = files.upload()
    nome_zip = next(iter(enviado))
    with zipfile.ZipFile(nome_zip) as z:
        z.extractall("/content")

pasta = next(p for p in ["/content/campo-real-dw", "/content"] if os.path.exists(os.path.join(p, "run_pipeline.py")))
os.chdir(pasta)
sys.path.insert(0, pasta)
print("Projeto em", pasta)

## 2. Instalar as dependências

In [ ]:
!pip install -q -r requirements.txt

## 3. Conexão com o banco
No Neon: **Dashboard → Connect** → copie a string que começa com `postgresql://`. Ela é digitada escondida e gravada só no `.env` desta sessão (não vai para o GitHub).

In [ ]:
from getpass import getpass

url = getpass("Cole a string de conexão do PostgreSQL: ").strip()
with open(".env", "w") as f:
    f.write(f"DATABASE_URL={url}\n")

from src import db
print(db.escalar("SELECT version()"))

## 4. (Opcional) Criar o banco de origem
Só rode se as tabelas `usina`, `medicao_geracao` etc. **ainda não existem** no banco. Mude para `True` e envie o `campo_real_energia_postgres.sql` da professora.

In [ ]:
import re

CARREGAR_ORIGEM = False

if CARREGAR_ORIGEM:
    enviado = files.upload()
    texto = enviado[next(iter(enviado))].decode("utf-8-sig")
    texto = "\n".join(l for l in texto.splitlines() if not l.strip().startswith("\\"))
    texto = re.sub(r"(?is)\b(DROP|CREATE)\s+DATABASE\b[^;]*;", "", texto)
    with db.conexao_crua() as con:
        with con.cursor() as cur:
            cur.execute(texto)
    print("Origem carregada.")

print(db.consultar("""SELECT 'usina' AS tabela, count(*) FROM usina
UNION ALL SELECT 'medicao_geracao', count(*) FROM medicao_geracao
UNION ALL SELECT 'parada_usina', count(*) FROM parada_usina"""))

## 5. Rodar o pipeline completo
Extração do PLD (CCEE) → qualidade → extração do clima (Open-Meteo) → qualidade do clima → carga do DW → relatório. Leva alguns minutos, a maior parte na API da CCEE.

In [ ]:
!python run_pipeline.py

## 6. Conferir o log de execuções

In [ ]:
db.consultar("""SELECT id_execucao, processo, fonte, status, linhas_lidas, linhas_gravadas,
       round(extract(epoch FROM fim - inicio)::numeric, 1) AS segundos, left(mensagem_erro, 120) AS mensagem
  FROM controle.log_execucao ORDER BY id_execucao""")

## 7. Números para a apresentação
Copie esta saída e mande no chat para atualizar os slides com os valores reais.

In [ ]:
import pandas as pd
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 20)

consultas = {
    "Geração e receita 2025": """SELECT round(sum(energia_mwh)/1000,1) AS geracao_gwh, round(sum(receita_rs)/1e6,2) AS receita_mi,
        round(sum(receita_rs)/sum(energia_mwh) FILTER (WHERE receita_rs IS NOT NULL),2) AS preco_capturado
        FROM dw_campo_real.fato_geracao_horaria WHERE sk_usina <> -1""",
    "Por fonte": """SELECT u.fonte, round(sum(f.energia_mwh)/1000,1) AS gwh, round(sum(f.receita_rs)/1e6,2) AS receita_mi,
        round(sum(f.receita_rs)/sum(f.energia_mwh),2) AS preco_capt, round(sum(f.receita_rs)/sum(f.receita_perfil_plano_rs),3) AS razao_captura,
        round(100*sum(f.energia_mwh)/sum(f.capacidade_mwh),1) AS fc_pct
        FROM dw_campo_real.fato_geracao_horaria f JOIN dw_campo_real.dim_usina u USING (sk_usina)
        WHERE f.receita_rs IS NOT NULL GROUP BY 1 ORDER BY razao_captura DESC""",
    "PLD por submercado e ano": """SELECT t.ano, s.cod_submercado, round(avg(p.pld_rs_mwh),2) AS pld_medio, min(p.pld_rs_mwh) AS minimo,
        max(p.pld_rs_mwh) AS maximo, round(100.0*avg(p.flag_piso::int),1) AS pct_horas_piso, max(t.ano_mes) AS ultimo_mes
        FROM dw_campo_real.fato_pld_horario p JOIN dw_campo_real.dim_tempo t USING (sk_tempo)
        JOIN dw_campo_real.dim_submercado s USING (sk_submercado) GROUP BY 1,2 ORDER BY 1,2""",
    "PLD médio por hora (2025)": """SELECT t.hora, round(avg(p.pld_rs_mwh),0) AS pld FROM dw_campo_real.fato_pld_horario p
        JOIN dw_campo_real.dim_tempo t USING (sk_tempo) WHERE t.ano = 2025 GROUP BY 1 ORDER BY 1""",
    "Paradas": """SELECT p.tipo_parada, u.cod_usina, sum(p.horas_indisponiveis) AS horas, round(sum(p.energia_esperada_mwh)) AS mwh,
        round(sum(p.receita_perdida_rs)/1e3,1) AS receita_mil FROM dw_campo_real.fato_parada_horaria p
        JOIN dw_campo_real.dim_usina u USING (sk_usina) GROUP BY 1,2 ORDER BY receita_mil DESC""",
    "Clima x geração": """SELECT u.fonte, round(corr(v.radiacao_w_m2, v.energia_mwh)::numeric,3) AS corr_radiacao,
        round(corr(v.vento_100m_ms, v.energia_mwh)::numeric,3) AS corr_vento
        FROM dw_campo_real.vw_geracao_clima v JOIN dw_campo_real.dim_usina u USING (sk_usina) GROUP BY 1""",
    "Impacto da limpeza": """SELECT indicador, recorte, round(antes,2) AS antes, round(depois,2) AS depois, round(pct_variacao,2) AS pct
        FROM dw_campo_real.agg_impacto_limpeza WHERE recorte_tipo IN ('TOTAL','FONTE') ORDER BY 1,2""",
}
for titulo, sql in consultas.items():
    print(f"\n===== {titulo} =====")
    print(db.consultar(sql).to_string(index=False))

## 8. Baixar o relatório de qualidade gerado

In [ ]:
files.download("docs/relatorio_qualidade.md")